## <center> **Generación modelo estrella con spark**


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.functions import col
from pyspark.sql.functions import year, month, dayofmonth, date_format, dayofweek
import os

Observación:

- Requiere instalación de  html[connector/j 8.0.33](https://dev.mysql.com/downloads/file/?id=552110):
    - Decargar el .zip
    - descomprimirlo en la ruta: 'C:\spark\jars\mysql-connector-j-8.0.33.jar'


In [2]:
jar = "C:\\spark\\jars\\mysql-connector-j-8.0.33.jar"

spark = SparkSession.builder \
    .appName("ETL Trips DW") \
    .config("spark.driver.extraClassPath", jar) \
    .config("spark.executor.extraClassPath", jar) \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()
    
    

# conexión MySQL
#url = "jdbc:mysql://localhost:3306/ecobicis"
url = "jdbc:mysql://localhost:3306/ecobicis?useSSL=false&serverTimezone=UTC&allowPublicKeyRetrieval=true"
#url = "jdbc:mysql://localhost:3306/ecobicis?rewriteBatchedStatements=true"
properties = {
    "user": "root",
    "password": "astro123",
    "driver": "com.mysql.cj.jdbc.Driver"
}

In [3]:
query = """
(
 SELECT 
    Nombre_Archivo,
    YEAR(STR_TO_DATE(Fecha_Retiro, '%Y-%m-%d')) AS anio,
    MONTH(STR_TO_DATE(Fecha_Retiro, '%Y-%m-%d')) AS mes,
    CAST(CAST(Ciclo_Estacion_Retiro AS SIGNED) AS CHAR) AS retiro,
    CAST(CAST(Ciclo_Estacion_Arribo AS SIGNED) AS CHAR) AS arribo,
    COUNT(*) AS total_viajes
 FROM viajes
 WHERE STR_TO_DATE(Fecha_Retiro, '%Y-%m-%d') IS NOT NULL
   AND Ciclo_Estacion_Retiro IS NOT NULL
   AND Ciclo_Estacion_Arribo IS NOT NULL
 GROUP BY 
    Nombre_Archivo,
    anio,
    mes,
    retiro,
    arribo
) t
"""

df = spark.read \
    .format("jdbc") \
    .option("url", url) \
    .option("dbtable", query) \
    .option("user", "root") \
    .option("password", "astro123") \
    .load()



In [4]:
df.toPandas()

,Nombre_Archivo,anio,mes,retiro,arribo,total_viajes
0,2010-02-feb.csv,2010,2,10,10,1
1,2010-02-feb.csv,2010,2,10,85,4
2,2010-02-feb.csv,2010,2,11,10,2
3,2010-02-feb.csv,2010,2,11,11,8
4,2010-02-feb.csv,2010,2,11,12,2
...,...,...,...,...,...,...
16364927,ecobici_2024_enero.csv,2024,1,99,95,54
16364928,ecobici_2024_enero.csv,2024,1,99,96,51
16364929,ecobici_2024_enero.csv,2024,1,99,97,37
16364930,ecobici_2024_enero.csv,2024,1,99,98,68


In [ ]:
df_clean = df \
    .withColumn("Genero_Usuario", upper(trim(col("Genero_Usuario")))) \
    .withColumn("Genero_Usuario", coalesce(col("Genero_Usuario"), lit("U"))) \
    .withColumn("Genero_Usuario", when(col("Genero_Usuario") == "", "U").otherwise(col("Genero_Usuario"))) \
    .withColumn("Edad_Usuario", coalesce(col("Edad_Usuario"), lit(-1))) \
    .withColumn("Edad_Usuario", col("Edad_Usuario").cast("int")) \
    .withColumn("Edad_Usuario",
        when((col("Edad_Usuario") < 0) | (col("Edad_Usuario") > 120), -1)
        .otherwise(col("Edad_Usuario"))
    ) \
    .withColumn("start_code", trim(col("Ciclo_Estacion_Retiro"))) \
    .withColumn("end_code", trim(col("Ciclo_Estacion_Arribo"))) \
    .filter(
        col("Bici").isNotNull() &
        col("start_code").isNotNull() &
        col("end_code").isNotNull() &
        col("Fecha_Retiro").isNotNull() &
        col("Fecha_Arribo").isNotNull() &
        col("Hora_Retiro").isNotNull() &
        col("Hora_Arribo").isNotNull()
    ) \
    .withColumn("start_time", col("Hora_Retiro")) \
    .withColumn("end_time", col("Hora_Arribo")) \
    .withColumn("trip_duration_sec", unix_timestamp("Hora_Arribo") - unix_timestamp("Hora_Retiro"))

**dim_user**

In [ ]:
dim_user = df_clean \
    .select(
        col("Genero_Usuario").alias("gender"),
        col("Edad_Usuario").alias("age")
    ) \
    .withColumn("gender", col("gender").substr(1,1)) \
    .withColumn("gender",
        when(col("gender").isin("M","F"), col("gender")).otherwise("U")
    ) \
    .withColumn("age", col("age").cast("short")) \
    .dropDuplicates()

**dim_bike**

In [ ]:
dim_bike = df_clean \
    .select(col("Bici").alias("bike_id")) \
    .dropDuplicates()

**dim_station**

In [ ]:
dim_station = df_clean \
    .select(col("start_code").alias("code")) \
    .union(df_clean.select(col("end_code").alias("code"))) \
    .filter(col("code") != "") \
    .dropDuplicates()

**dim_time**

In [ ]:
dim_time = df_clean \
    .select(col("Fecha_Retiro").alias("date")) \
    .union(df_clean.select(col("Fecha_Arribo").alias("date"))) \
    .dropDuplicates() \
    .withColumn("year", year("date").cast("short")) \
    .withColumn("month", month("date").cast("byte")) \
    .withColumn("day", dayofmonth("date").cast("byte")) \
    .withColumn("month_name", date_format("date", "MMM")) \
    .withColumn("day_name", date_format("date", "EEE")) \
    .withColumn("day_of_week", dayofweek("date").cast("byte"))

In [ ]:
def write_safe(df, table):
    df.coalesce(1).write \
        .format("jdbc") \
        .option("url", url) \
        .option("dbtable", table) \
        .option("user", "root") \
        .option("password", "astro123") \
        .option("driver", "com.mysql.cj.jdbc.Driver") \
        .option("partitionColumn", "Bici") \
        .option("lowerBound", 1) \
        .option("upperBound", 10000000) \
        .option("numPartitions", 4) \
        .option("batchsize", 10000) \
        .mode("append") \
        .save()

In [ ]:
properties = {
    "user": "root",
    "password": "astro123",
    "driver": "com.mysql.cj.jdbc.Driver"
}

In [ ]:
write_safe(dim_user, "dim_user")

In [ ]:
write_safe(dim_user, "dim_user")
write_safe(dim_bike, "dim_bike")
write_safe(dim_station, "dim_station")
write_safe(dim_time, "dim_time")

**df_fact**

In [ ]:
df_fact = df_clean \
    .withColumn("start_time", concat_ws(" ", col("Fecha_Retiro"), col("Hora_Retiro"))) \
    .withColumn("end_time", concat_ws(" ", col("Fecha_Arribo"), col("Hora_Arribo"))) \
    .withColumn("start_time", col("start_time").cast("timestamp")) \
    .withColumn("end_time", col("end_time").cast("timestamp")) \
    .withColumn("trip_duration_sec", unix_timestamp("end_time") - unix_timestamp("start_time"))

joins

In [ ]:
dim_user_db = spark.read.jdbc(url, "dim_user", properties)
dim_bike_db = spark.read.jdbc(url, "dim_bike", properties)
dim_station_db = spark.read.jdbc(url, "dim_station", properties)

trips = df_fact \
    .join(dim_user_db,
          (df_fact.Genero_Usuario == dim_user_db.gender) &
          (df_fact.Edad_Usuario == dim_user_db.age)) \
    .join(dim_bike_db, df_fact.Bici == dim_bike_db.bike_id) \
    .join(dim_station_db, df_fact.Ciclo_Estacion_Retiro == dim_station_db.code) \
    .withColumnRenamed("station_id", "start_station_id") \
    .join(dim_station_db, df_fact.Ciclo_Estacion_Arribo == dim_station_db.code) \
    .withColumnRenamed("station_id", "end_station_id")

In [ ]:
trips_final = trips.select(
    col("Fecha_Retiro").alias("start_date"),
    col("user_id"),
    col("bike_id"),
    col("start_station_id"),
    col("end_station_id"),
    col("start_time"),
    col("end_time"),
    col("trip_duration_sec"),
    year("Fecha_Retiro").alias("start_year"),
    month("Fecha_Retiro").alias("start_month"),
    dayofmonth("Fecha_Retiro").alias("start_day")
)

In [ ]:
write_safe(trips_final, "trips")